# Experiment 3 — GRU

In [1]:
import pandas as pd

df_clean = pd.read_csv("../data/cleaned_data.csv")

df_clean["date"] = pd.to_datetime(df_clean["date"])
df_clean = df_clean.iloc[:, 1:3]
df_clean.head()

,date,demand
0,2015-01-01,99635.030
1,2015-01-02,129606.010
2,2015-01-03,142300.540
3,2015-01-04,104330.715
4,2015-01-05,118132.200


In [2]:
import torch
import torch.nn as nn

torch.manual_seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

## Chronological Train/test split

In [3]:
import numpy as np

# train-test split

df = df_clean.set_index("date")
df = df.sort_index()

# demand is the only feature
values = df["demand"].values

n = len(values)

train_end = int(0.8 * n)

train_values = values[:train_end]
test_values = values[train_end:]

print(f"Train: {len(train_values)}")
print(f"Test: {len(test_values)}")

Train: 1684
Test: 422


## Data Transformation

In [4]:
# min-max scaling (fit on train only, applied to both)
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()

train_scaled = scaler.fit_transform(
    train_values.reshape(-1, 1)
)
test_scaled = scaler.transform(
    test_values.reshape(-1, 1)
)
print(np.shape(train_scaled))
print(np.shape(test_scaled))

(1684, 1)
(422, 1)


## Create the sliding window sequences

Use the previous `sequence_length` days of demand to predict the next day's demand.

In [5]:
SEQUENCE_LENGTH = 7


def create_sequences(values: np.ndarray, sequence_length: int):
    """
    Create sliding-window sequences for next-day forecasting.

    Example with sequence_length=7:

        X = days 1-7
        y = day 8

        X = days 2-8
        y = day 9
    """

    X = []
    y = []

    for i in range(len(values) - sequence_length):
        X.append(values[i : i + sequence_length])
        y.append(values[i + sequence_length])

    return np.array(X), np.array(y)


X_train, y_train = create_sequences(train_scaled, SEQUENCE_LENGTH)
X_test, y_test = create_sequences(test_scaled, SEQUENCE_LENGTH)

print(np.shape(X_train))
print(np.shape(y_train))

(1677, 7, 1)
(1677, 1)


## Convert to PyTorch Tensors

In [6]:
# nn.GRU expects: (batch_size, sequence_length, input_size)
# create_sequences already preserves the feature dim from the (N, 1) scaled arrays,
# so X is already (N, seq_len, 1) and y is already (N, 1).
X_train = torch.tensor(X_train, dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.float32)

X_test = torch.tensor(X_test, dtype=torch.float32)
y_test = torch.tensor(y_test, dtype=torch.float32)

print(X_train.shape)
print(y_train.shape)
print(X_test.shape)
print(y_test.shape)

torch.Size([1677, 7, 1])
torch.Size([1677, 1])
torch.Size([415, 7, 1])
torch.Size([415, 1])


## Data Loader

In [7]:
from torch.utils.data import DataLoader, TensorDataset

BATCH_SIZE = 64

train_dataset = TensorDataset(X_train, y_train)
test_dataset = TensorDataset(X_test, y_test)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

## Create the model

Uses PyTorch's built-in `nn.GRU` (vs. the from-scratch RNN in Experiment 1 and `nn.LSTM` in Experiment 2). Many-to-one: the final layer's final hidden state is fed to a linear output layer to make one prediction per sequence. GRUs have no separate cell state — only a single hidden state, fewer gates than an LSTM (2 vs. 3), and thus fewer parameters.

In [8]:
class ElectricityDemandGRU(nn.Module):
    """GRU built on PyTorch's built-in nn.GRU."""

    def __init__(self, input_size: int, hidden_size: int, output_size: int, num_layers: int = 1):
        """
        Args:
            input_size: How many features come into the GRU at one timestep.
            hidden_size: How much information the GRU keeps in its memory.
            output_size: How many values you want the model to predict.
            num_layers: How many stacked GRU layers.
        """
        super().__init__()

        self.hidden_size = hidden_size
        self.num_layers = num_layers

        self.gru = nn.GRU(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
        )
        self.fc = nn.Linear(hidden_size, output_size)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x: Input sequence with shape (batch_size, sequence_length, input_size).

        Returns:
            One prediction per sequence with shape (batch_size, output_size).
        """

        # gru_out: (batch_size, sequence_length, hidden_size)
        # h_n: (num_layers, batch_size, hidden_size) — final hidden state per layer
        _, h_n = self.gru(x)

        # many-to-one: use the final layer's final hidden state
        last_hidden = h_n[-1]

        return self.fc(last_hidden)

In [9]:
INPUT_SIZE = train_scaled.shape[-1]   # 1
OUTPUT_SIZE = train_scaled.shape[-1]  # 1
HIDDEN_SIZE = 32
NUM_LAYERS = 1

EPOCHS = 10

model = ElectricityDemandGRU(INPUT_SIZE, HIDDEN_SIZE, OUTPUT_SIZE, NUM_LAYERS).to(device)
criterion = nn.L1Loss()  # pytorch's MAE
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)


def mae(y_true, y_pred):
    return np.mean(np.abs(y_true - y_pred))


def rmse(y_true, y_pred):
    return np.sqrt(np.mean((y_true - y_pred) ** 2))


for name, param in model.named_parameters():
    print(name, param.shape)

gru.weight_ih_l0 torch.Size([96, 1])
gru.weight_hh_l0 torch.Size([96, 32])
gru.bias_ih_l0 torch.Size([96])
gru.bias_hh_l0 torch.Size([96])
fc.weight torch.Size([1, 32])
fc.bias torch.Size([1])


## Training Loop

In [10]:
def train_one_epoch(model, train_loader, criterion, optimizer, device):

    model.train()

    total_loss = 0

    for x_batch, y_batch in train_loader:
        x_batch = x_batch.to(device)
        y_batch = y_batch.to(device)

        optimizer.zero_grad()

        y_pred = model(x_batch)

        loss = criterion(y_pred, y_batch)

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)

    return avg_loss

## Train for multiple Epochs (single run, sequence_length=7)

In [11]:
for epoch in range(EPOCHS):

    train_loss = train_one_epoch(
        model=model,
        train_loader=train_loader,
        criterion=criterion,
        optimizer=optimizer,
        device=device
    )

    print(f"Epoch {epoch + 1}/{EPOCHS} | Loss: {train_loss:.4f}")

Epoch 1/10 | Loss: 0.2659
Epoch 2/10 | Loss: 0.1372
Epoch 3/10 | Loss: 0.1281
Epoch 4/10 | Loss: 0.1241
Epoch 5/10 | Loss: 0.1227
Epoch 6/10 | Loss: 0.1199
Epoch 7/10 | Loss: 0.1180
Epoch 8/10 | Loss: 0.1156
Epoch 9/10 | Loss: 0.1131
Epoch 10/10 | Loss: 0.1106


## Evaluate

Note: the training loss printed above is MAE on **min-max scaled** demand, not real units. Inverse-transform predictions back to the original demand scale before computing MAE/RMSE.

In [12]:
def evaluate_model(model, data_loader, scaler, device):
    """
    Run the model on a data loader and return predictions/targets
    in the ORIGINAL demand scale, plus real-unit MAE/RMSE.
    """

    model.eval()

    all_preds = []
    all_targets = []

    with torch.no_grad():
        for x_batch, y_batch in data_loader:
            x_batch = x_batch.to(device)

            y_pred = model(x_batch)

            all_preds.append(y_pred.cpu())
            all_targets.append(y_batch.cpu())

    preds_scaled = torch.cat(all_preds).numpy()
    targets_scaled = torch.cat(all_targets).numpy()

    # undo min-max scaling to get back to real demand units
    preds = scaler.inverse_transform(preds_scaled)
    targets = scaler.inverse_transform(targets_scaled)

    test_mae = mae(targets, preds)
    test_rmse = rmse(targets, preds)

    return preds, targets, test_mae, test_rmse


preds, targets, test_mae, test_rmse = evaluate_model(model, test_loader, scaler, device)

print(f"Test MAE:  {test_mae:,.2f} demand units")
print(f"Test RMSE: {test_rmse:,.2f} demand units")

Test MAE:  8,473.12 demand units
Test RMSE: 10,614.45 demand units


## Sweep: sequence length vs. performance

Repeat the full pipeline (sequence creation → tensors → loader → fresh model → train → evaluate) for `sequence_length` in `[1, 7, 14, 30, 60]`, and record MAE, RMSE, training time, and inference latency for each — same methodology as Experiment 1 (vanilla RNN) and Experiment 2 (LSTM).

In [13]:
import time


def run_experiment(sequence_length: int, hidden_size: int = HIDDEN_SIZE,
                    num_layers: int = NUM_LAYERS, epochs: int = EPOCHS,
                    batch_size: int = BATCH_SIZE, lr: float = 0.001, seed: int = 42):
    """
    Build sequences, train a fresh ElectricityDemandGRU, and evaluate it
    for a given sequence_length. Returns a dict of results.

    train_scaled / test_scaled / scaler / device are reused from the
    outer scope (already fit on the chronological train/test split).
    """

    torch.manual_seed(seed)

    # 1. sliding-window sequences for this sequence_length
    X_tr, y_tr = create_sequences(train_scaled, sequence_length)
    X_te, y_te = create_sequences(test_scaled, sequence_length)

    # 2. tensors — already (N, seq_len, 1) / (N, 1), no unsqueeze needed
    X_tr = torch.tensor(X_tr, dtype=torch.float32)
    y_tr = torch.tensor(y_tr, dtype=torch.float32)
    X_te = torch.tensor(X_te, dtype=torch.float32)
    y_te = torch.tensor(y_te, dtype=torch.float32)

    # 3. loaders
    train_loader_exp = DataLoader(TensorDataset(X_tr, y_tr), batch_size=batch_size, shuffle=False)
    test_loader_exp = DataLoader(TensorDataset(X_te, y_te), batch_size=batch_size, shuffle=False)

    # 4. fresh model per run so results aren't contaminated by earlier training
    model_exp = ElectricityDemandGRU(INPUT_SIZE, hidden_size, OUTPUT_SIZE, num_layers).to(device)
    criterion_exp = nn.L1Loss()
    optimizer_exp = torch.optim.Adam(model_exp.parameters(), lr=lr)

    # 5. train, timing the whole training run
    train_start = time.perf_counter()
    for _ in range(epochs):
        train_one_epoch(
            model=model_exp,
            train_loader=train_loader_exp,
            criterion=criterion_exp,
            optimizer=optimizer_exp,
            device=device,
        )
    train_time = time.perf_counter() - train_start

    # 6. evaluate + inference latency (ms per sample, single-sample batches)
    single_loader = DataLoader(TensorDataset(X_te, y_te), batch_size=1, shuffle=False)
    model_exp.eval()
    with torch.no_grad():
        infer_start = time.perf_counter()
        for x_batch, _ in single_loader:
            model_exp(x_batch.to(device))
        infer_time = time.perf_counter() - infer_start
    latency_ms = (infer_time / len(single_loader.dataset)) * 1000

    _, _, test_mae, test_rmse = evaluate_model(model_exp, test_loader_exp, scaler, device)

    return {
        "sequence_length": sequence_length,
        "mae": test_mae,
        "rmse": test_rmse,
        "training_time_s": train_time,
        "inference_latency_ms": latency_ms,
    }

In [14]:
SEQUENCE_LENGTHS = [1, 7, 14, 30, 60]

results = [run_experiment(seq_len) for seq_len in SEQUENCE_LENGTHS]

results_df = pd.DataFrame(results).set_index("sequence_length")
results_df

,mae,rmse,training_time_s,inference_latency_ms
sequence_length,,,,
1,8906.745117,10950.436523,3.089812,2.565780
7,8473.120117,10614.452148,2.796977,1.394810
14,8331.573242,10421.300781,2.189217,0.934122
30,8322.538086,10420.489258,1.735753,1.035174
60,8428.742188,10580.535156,3.339650,1.572851
